# NumPy：発生間隔とビン境界

対応する本文: [numpy.tex](../chapters/numpy.tex)。各問を「考え方 → 実装 → 確認」の順で解説する。上から全セルを実行すると、入力の作成から結果の検証までを再現できる。生成物は専用の出力先へ保存する。

In [1]:
import sys
from pathlib import Path

# リポジトリ直下と notebooks/ のどちらから起動しても同じ場所を使う。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "main.tex").exists() and (p / "chapters").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("教材のフォルダー内で Notebook を起動してください。")
WORK = ROOT / "notebooks" / "_outputs" / "05_numpy"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("出力先:", WORK.relative_to(ROOT))

Python 3.12.14
出力先: notebooks/_outputs/05_numpy


## 問1：ソートしてから差を取る

本文の人工時刻列を使う。`np.sort` は整列した配列を返し、`np.diff` は隣接する2要素の差を返す。6個の時刻から5個の間隔ができる。

In [2]:
import numpy as np

original = np.array([720.0, 0.0, 60.0, 4200.0, 180.0, 780.0])
times = np.sort(original)
dt = np.diff(times)
np.testing.assert_array_equal(times, [0, 60, 180, 720, 780, 4200])
np.testing.assert_array_equal(dt, [60, 120, 540, 60, 3420])
print("時刻:", times, times.shape, times.dtype)
print("間隔:", dt, dt.shape, dt.dtype)

時刻: [   0.   60.  180.  720.  780. 4200.] (6,) float64
間隔: [  60.  120.  540.   60. 3420.] (5,) float64


実カタログでは、UTC の時刻を同じ基準からの秒へ変換する。次の関数へ CSV のパスを渡すと同じ処理を実行できる。

In [3]:
import csv
from datetime import datetime


def intervals_from_csv(path):
    with Path(path).open(encoding="utf-8", newline="") as f:
        stamps = [datetime.fromisoformat(row["time"]) for row in csv.DictReader(f)]
    if len(stamps) < 2:
        raise ValueError("発生間隔の計算には2件以上が必要です。")
    origin = min(stamps)
    seconds = np.array([(stamp - origin).total_seconds() for stamp in stamps], dtype=np.float64)
    return np.diff(np.sort(seconds))

## 問2：半開区間とヒストグラムを区別する

指定された3区間はすべて右端を含めない。一方、`np.histogram` は最後のビンだけ右端も含む。通常の入力では差が見えないため、3600秒ちょうどの値も別に試す。

In [4]:
interval_counts = [np.count_nonzero((dt >= low) & (dt < high))
                   for low, high in [(0, 60), (60, 600), (600, 3600)]]
edges = np.arange(0.0, 3600.0 + 600.0, 600.0)
counts, bin_edges = np.histogram(dt, bins=edges)
k = counts.argmax()
print("平均 [s]:", dt.mean(), "正の間隔:", np.count_nonzero(dt > 0))
print("3区間:", interval_counts)
print("度数:", counts, "最頻ビン:", bin_edges[k:k+2])
assert dt.mean() == 840 and np.count_nonzero(dt > 0) == 5
assert interval_counts == [0, 4, 1]
np.testing.assert_array_equal(counts, [4, 0, 0, 0, 0, 1])
assert len(bin_edges) == len(counts) + 1
boundary = np.array([3600.0])
assert np.count_nonzero((boundary >= 600) & (boundary < 3600)) == 0
assert np.histogram(boundary, bins=edges)[0][-1] == 1

平均 [s]: 840.0 正の間隔: 5
3区間: [np.int64(0), np.int64(4), np.int64(1)]
度数: [4 0 0 0 0 1] 最頻ビン: [  0. 600.]


## 問3：固定幅と対数幅を比較する

境界を30個作ると29ビンになる。対数ビンの幅は一定ではない。正の値を対象とし、0や範囲外の値が何件あるかも確認する。横軸を対数にする操作と、ビンを対数間隔にする操作は別である。

In [5]:
log_edges = np.logspace(np.log10(1), np.log10(3600), 30)
log_edges[-1] = 3600.0  # 丸め誤差によって指定した上端を除外しないようにする。
for name, bins in [("固定幅", edges), ("対数幅", log_edges)]:
    freq, _ = np.histogram(dt, bins=bins)
    print(name, "ビン数:", len(freq), "空ビン:", np.count_nonzero(freq == 0),
          "最小・最大幅:", np.diff(bins).min(), np.diff(bins).max(),
          "範囲外:", np.count_nonzero((dt < bins[0]) | (dt > bins[-1])))
    assert freq.sum() == len(dt)

固定幅 ビン数: 6 空ビン: 4 最小・最大幅: 600.0 600.0 範囲外: 0
対数幅 ビン数: 29 空ビン: 25 最小・最大幅: 0.3262674794960523 885.6154164558807 範囲外: 0


### 発展：日本近海と全球を比較する

日本近海を経度122–154度、緯度20–50度と定める。下のコードは取得済みの全球カタログから日本近海を抽出するので、取得期間と更新時点をそろえられる。実データのパスを設定した場合だけ実行する。地域内を抽出してから時刻差を再計算する点に注意する。全球の間隔配列を緯度で絞る操作とは異なる。

件数、期間、マグニチュード下限や種別をそろえて分布を比較する。観測範囲や検出能力が異なるので、間隔の差をそのまま地域固有の物理過程の差とは解釈しない。

In [6]:
REAL_CSV = None  # 例: ROOT / "earthquake_practice/raw/usgs_global_2024_01.csv"
if REAL_CSV is not None:
    import pandas as pd
    catalog = pd.read_csv(REAL_CSV)
    japan = catalog.loc[catalog.longitude.between(122, 154) & catalog.latitude.between(20, 50)]
    japan_path = WORK / "usgs_japan.csv"
    japan.to_csv(japan_path, index=False)
    global_dt = intervals_from_csv(REAL_CSV)
    japan_dt = intervals_from_csv(japan_path)
    print("全球:", len(global_dt), "間隔; 日本近海:", len(japan_dt), "間隔")
else:
    print("地域比較は取得済みの実カタログを指定して実行します。")

地域比較は取得済みの実カタログを指定して実行します。
